# Supplementary figure: Count imputation

Run after the training and evaluation commands in `bash/paper/`. SCENE outputs use the `scLDM` names referenced below.


In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment_scene.yaml").exists())
os.chdir(PROJECT_ROOT / "notebooks" / "figures")
for folder in ("fig_1", "fig_2", "fig_3", "fig_4", "fig_5", "fig_6", "app", "qc"):
    Path("output", folder).mkdir(parents=True, exist_ok=True)


In [ ]:
import numpy as np
import pandas as pd

# SCENE
masked_counts_scene_zip = np.load('../../results/imputation/pbmc_cite_seq_SCENE_zip/held_out_counts.npz')
pred_counts_scene_zip = np.load('../../results/imputation/pbmc_cite_seq_SCENE_zip/model_predictions.npz')

masked_counts_scene_poisson = np.load('../../results/imputation/pbmc_cite_seq_SCENE_poisson/held_out_counts.npz')
pred_counts_scene_poisson = np.load('../../results/imputation/pbmc_cite_seq_SCENE_poisson/model_predictions.npz')

# scVI
masked_counts_scvi = np.load('../../results/imputation/pbmc_cite_seq_scVI/held_out_counts.npz')
pred_counts_scvi = np.load('../../results/imputation/pbmc_cite_seq_scVI/model_predictions.npz')

true_vals = masked_counts_scene_poisson['heldout_count']

imputed_scldm = pred_counts_scene_zip['SCLDM'] #/ (-np.expm1(-pred_counts_scene_poisson['SCLDM']))
imputed_scldm_poisson = pred_counts_scene_poisson['SCLDM'] #/ (-np.expm1(-pred_counts_scene_poisson['SCLDM']))
imputed_scvi  = pred_counts_scvi['SCVI']
for other in (masked_counts_scene_zip, masked_counts_scvi):
    for key in ("row", "col", "heldout_count"):
        np.testing.assert_array_equal(masked_counts_scene_poisson[key], other[key])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import issparse
from sklearn.metrics import mean_poisson_deviance


# --- Imputation metrics ---
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def pearsonr(y_true, y_pred):
    # Pearson correlation coefficient
    y_true_m = y_true - y_true.mean()
    y_pred_m = y_pred - y_pred.mean()
    return np.dot(y_true_m, y_pred_m) / np.sqrt(np.sum(y_true_m**2)*np.sum(y_pred_m**2))

def spearmanr(y_true, y_pred):
    # Spearman rank correlation
    from scipy.stats import rankdata
    return pearsonr(rankdata(y_true), rankdata(y_pred))

metrics = {}
for name, pred in [('scLDM', imputed_scldm), ('scLDM_poisson', imputed_scldm_poisson), ('scVI', imputed_scvi)]:
    metrics[name] = {
        'RMSE': rmse(true_vals, pred),
        'MAE': mae(true_vals, pred),
        'Pearson': pearsonr(true_vals, pred),
        'Spearman': spearmanr(true_vals, pred),
        'PoissonDeviance': mean_poisson_deviance(true_vals, pred),
    }

pd.DataFrame(metrics).T.to_csv("output/app/imputation_metrics.csv")


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],

    "font.size": 6,
    "axes.titlesize": 7,
    "axes.labelsize": 6,
    "xtick.labelsize": 5.5,
    "ytick.labelsize": 5.5,
    "legend.fontsize": 5.5,

    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.minor.width": 0.4,
    "ytick.minor.width": 0.4,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "xtick.minor.size": 1.5,
    "ytick.minor.size": 1.5,

    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# --- Plotting ---
red  = "#D76155"   # SCENE
blue = "#6583DD"   # scVI

model_order = ["scVI", "scLDM"]
model_labels = {
    "scVI": "scVI",
    "scLDM": "SCENE",
}
model_colors = {
    "scVI": blue,
    "scLDM": red,
}

preds = {
    "scVI": imputed_scvi,
    "scLDM": imputed_scldm,
}

# --- Figure layout ---
fig = plt.figure(figsize=(6.4, 4.8), constrained_layout=True)

outer = fig.add_gridspec(
    2, 1,
    height_ratios=[1.3, 1],
)

gs_top = outer[0].subgridspec(
    1, 2,
    wspace=0.20,
)

gs_bot = outer[1].subgridspec(
    1, 3,
    wspace=0.20,
)

# common 1:1 line bounds
minv = min(
    true_vals.min(),
    imputed_scvi.min(),
    imputed_scldm.min(),
)
maxv = max(
    true_vals.max(),
    imputed_scvi.max(),
    imputed_scldm.max(),
)

# --- Top row: scatter plots ---
for i, model in enumerate(model_order):
    ax = fig.add_subplot(gs_top[0, i])

    ax.scatter(
        true_vals,
        preds[model],
        s=3,
        alpha=0.8,
        color=model_colors[model],
        rasterized=True,
    )

    ax.plot([minv, maxv], [minv, maxv], "--", linewidth=1, color="gray")

    ax.set_xlim(left=1)
    ax.set_ylim(bottom=1)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("True counts")
    if i == 0:
        ax.set_ylabel("Imputed counts")
    else:
        plt.setp(ax.get_yticklabels(), visible=False)

    ax.set_title(f"True vs Imputed ({model_labels[model]})")

# --- Shared bar settings ---
width = 0.30
offsets = {
    model: (i - (len(model_order) - 1) / 2) * width
    for i, model in enumerate(model_order)
}

legend_handles = []

# --- Bottom-left: RMSE + MAE ---
ax3 = fig.add_subplot(gs_bot[0, 0])

errs = ["RMSE", "MAE"]
x_err = np.arange(len(errs))

ax3.set_axisbelow(True)
ax3.yaxis.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)

for model in model_order:
    bars = ax3.bar(
        x_err + offsets[model],
        [metrics[model][m] for m in errs],
        width,
        label=model_labels[model],
        color=model_colors[model],
        edgecolor="k",
        linewidth=0.8,
    )
    legend_handles.append(bars[0])

ax3.set_xticks(x_err)
ax3.set_xticklabels(errs)
ax3.set_xlim(-0.5, len(errs) - 0.5)
ax3.set_ylabel("Error")
ax3.set_title("Imputation Errors")

# --- Bottom-middle: correlations ---
ax4 = fig.add_subplot(gs_bot[0, 1])

cors = ["Pearson", "Spearman"]
x_corr = np.arange(len(cors))

ax4.set_axisbelow(True)
ax4.yaxis.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)

for model in model_order:
    ax4.bar(
        x_corr + offsets[model],
        [metrics[model][m] for m in cors],
        width,
        label=model_labels[model],
        color=model_colors[model],
        edgecolor="k",
        linewidth=0.8,
    )

ax4.set_xticks(x_corr)
ax4.set_xticklabels(cors)
ax4.set_xlim(-0.5, len(cors) - 0.5)
ax4.set_ylabel("Correlation")
ax4.set_title("Imputation Correlations")
ax4.set_ylim(0, 1)

# --- Bottom-right: Poisson deviance ---
ax5 = fig.add_subplot(gs_bot[0, 2])

x_dev = np.array([0.5])

ax5.set_axisbelow(True)
ax5.yaxis.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)

for model in model_order:
    ax5.bar(
        x_dev + offsets[model],
        [metrics[model]["PoissonDeviance"]],
        width,
        label=model_labels[model],
        color=model_colors[model],
        edgecolor="k",
        linewidth=0.8,
    )

ax5.set_xticks(x_dev)
ax5.set_xticklabels(["Poisson deviance"])

# Match the visual category spacing of the two-metric panels.
ax5.set_xlim(-0.5, 1.5)

ax5.set_ylabel("Mean deviance")
ax5.set_title("Poisson Deviance")

# --- Shared legend ---
fig.legend(
    legend_handles,
    [model_labels[m] for m in model_order],
    frameon=False,
    ncol=2,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.03),
)

plt.savefig("output/app/pbmc_imputation_scvi_vs_scene.svg", dpi=600, bbox_inches="tight")
plt.savefig("output/app/pbmc_imputation_scvi_vs_scene.pdf", dpi=600, bbox_inches="tight")
plt.show()